# Diarização — SciTech

Este notebook **não faz transcrição** (isso é feito em `transcricao.ipynb`). Ele parte de:

- um `.wav` da reunião, em `audios_brutos/`
- o `.json` já transcrito pelo `transcricao.ipynb`, com o **mesmo nome base**, em `transcricoes/` (ex: `reuniao1.wav` + `reuniao1.json`)
- um **banco de vozes** em `banco_vozes/`, com uma pasta por pessoa contendo os `.wav` de amostra (guardados, para permitir recadastro/atualização depois) e o `embedding.pt` já calculado a partir deles — o pipeline principal só lê o `.pt`, nunca recalcula a partir do áudio

E devolve um **`.json` diarizado**: cada segmento da transcrição ganha um `speaker` (nome da pessoa, se identificada, ou `SPEAKER_XX`).

> Formato esperado do JSON de entrada (ajuste a seção 5 se o `transcricao.ipynb` gerar outro formato):
> ```json
> {"segments": [{"start": 0.0, "end": 2.3, "text": "..."}, ...]}
> ```

> **Token do Hugging Face:** nunca cole o token direto no código. Use `userdata.get("HF_TOKEN")` (Colab → ícone de chave 🔑 → Secrets).

## 1. Setup: Google Drive e pastas do projeto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PASTA_BASE = "/content/drive/MyDrive/projeto_diarizacao"

AUDIOS       = os.path.join(PASTA_BASE, "audios_brutos")       # .wav das reuniões
TRANSCRICOES = os.path.join(PASTA_BASE, "transcricoes")        # .json gerado pelo transcricao.ipynb
SAIDA        = os.path.join(PASTA_BASE, "resultados")          # .json diarizado (saída deste notebook)
BANCO_VOZES  = os.path.join(PASTA_BASE, "banco_vozes")         # uma pasta por pessoa: audios/ + embedding.pt
SCRIPTS      = os.path.join(PASTA_BASE, "scripts")             # .py auxiliares (já devem existir aqui, colocados manualmente no Drive)

for pasta in (AUDIOS, TRANSCRICOES, SAIDA, BANCO_VOZES, SCRIPTS):
    os.makedirs(pasta, exist_ok=True)

os.chdir(PASTA_BASE)

print("Diretório do projeto:", os.getcwd())
print(os.listdir(PASTA_BASE))

## 2. Instalação das dependências

Não precisa de `whisperx` aqui — a transcrição já vem pronta de outro notebook.

In [ ]:
!pip install -q -U pyannote.audio   # versão 4.x, exigida pelo community-1
!pip install -q speechbrain

## 3. Token do Hugging Face

Guarde o token em **Secrets** do Colab (ícone de chave 🔑) com o nome `HF_TOKEN`, e dê acesso a este notebook.

⚠️ Se em algum momento você colou o token direto em texto puro num notebook/script versionado no Git, revogue-o em https://huggingface.co/settings/tokens e gere um novo — token exposto no histórico do repositório conta como vazado mesmo depois de apagado.

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "Token não encontrado. Adicione HF_TOKEN em Secrets (ícone de chave) "
        "e dê acesso a este notebook antes de continuar."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN carregado com sucesso.")

## 4. Scripts auxiliares

Os `.py` (`etapa2b_diarizacao.py`, `etapa3_biometria.py`, `pipeline.py`, `cadastro_vozes.py`) já devem estar em `projeto_diarizacao/scripts/` no Drive — não são gerados por este notebook. Se a pasta estiver vazia, copie os arquivos para lá antes de continuar.

## 5. Entradas: áudio + transcrição

Coloque em `audios_brutos/` o `.wav` da reunião, e em `transcricoes/` o `.json` correspondente (mesmo nome base) gerado pelo `transcricao.ipynb`.

Pode subir pelo Google Drive direto, ou usar a célula abaixo para os dois de uma vez.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()  # selecione os .wav e/ou .json na janela que abrir

for nome in uploaded.keys():
    destino = AUDIOS if nome.endswith(".wav") else TRANSCRICOES
    shutil.move(nome, os.path.join(destino, nome))

print("Áudios:", os.listdir(AUDIOS))
print("Transcrições:", os.listdir(TRANSCRICOES))

## 6. Banco de vozes — cadastro e atualização

**Rode esta seção quando cadastrar uma pessoa nova ou adicionar mais amostras a uma pessoa já cadastrada.** O áudio de amostra é **copiado e guardado** em `banco_vozes/<nome>/audios/` (não é descartado), e o embedding é recalculado a partir de **todas** as amostras daquela pessoa (antigas + novas) e salvo em `banco_vozes/<nome>/embedding.pt`.

O pipeline principal (seção 8) nunca lê essa pasta de áudios — só o `embedding.pt` — então isso não é refeito a cada reunião. Você só roda essa seção quando cadastra ou atualiza alguém.

Coloque os `.wav` novos de uma pessoa em uma subpasta temporária (ex: `cadastro_temp/Maria Clara/audio1.wav`, `audio2.wav`, ...) e rode a célula abaixo.

In [ ]:
import os
import shutil

for nome in ["Leandro", "Maria Clara", "Pedro", "Samuel"]:
    pasta_audios = os.path.join(BANCO_VOZES, nome, "audios")
    pasta_temp = os.path.join("/tmp/cadastro_temp", nome)

    if os.path.exists(pasta_temp):
        shutil.rmtree(pasta_temp)
    shutil.copytree(pasta_audios, pasta_temp)

    cadastrar_ou_atualizar_pessoa(nome, pasta_temp, BANCO_VOZES)

In [ ]:
print("Pessoas cadastradas em", BANCO_VOZES, ":")
if os.path.isdir(BANCO_VOZES):
    for nome in sorted(os.listdir(BANCO_VOZES)):
        pasta_pessoa = os.path.join(BANCO_VOZES, nome)
        tem_embedding = os.path.isfile(os.path.join(pasta_pessoa, "embedding.pt"))
        pasta_audios = os.path.join(pasta_pessoa, "audios")
        qtd_audios = len(os.listdir(pasta_audios)) if os.path.isdir(pasta_audios) else 0
        status = "OK" if tem_embedding else "sem embedding.pt"
        print(f"  {nome}: {qtd_audios} amostra(s) guardada(s) [{status}]")

## 7. Rodar o pipeline de diarização

In [ ]:
!python scripts/pipeline.py

In [ ]:
import torch, torchaudio
print("torch:", torch.__version__)
print("torchaudio:", torchaudio.__version__)

## 8. Conferir o JSON diarizado

In [ ]:
print("Arquivos gerados em", SAIDA, ":")
print(sorted(os.listdir(SAIDA)))

In [ ]:
import json

arquivos = sorted(f for f in os.listdir(SAIDA) if f.endswith(".json"))

if arquivos:
    with open(os.path.join(SAIDA, arquivos[0]), "r", encoding="utf-8") as f:
        dados = json.load(f)
    print(json.dumps(dados, ensure_ascii=False, indent=2)[:2000])
else:
    print("Rode o pipeline primeiro (seção 8) para gerar resultados.")

## 9. (Opcional) Baixar os resultados em .zip

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("resultados", 'zip', SAIDA)
files.download("resultados.zip")